# 03. 기회영역 분석
- 만족도(감성분석) + 중요도(언급량) → Opportunity Area
- Underserved Area = 핵심 기회영역

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
BASE_PATH = '/content/drive/Othercomputers/LG DX 노트북/LG_Dx_School/CX 프로젝트/가전 구독/가전 구독'
os.chdir(BASE_PATH)
!pwd

In [ ]:
!pip install -q new_value_analysis

### 0. 모든 Actor Action 데이터 로딩 및 통합

In [ ]:
import pandas as pd
import glob

df_list = []
pkl_files = sorted(glob.glob('./cluster_df_*_action.pkl'))
print(f'로딩할 파일: {pkl_files}')

for path in pkl_files:
    df_list.append(pd.read_pickle(path))

df = pd.concat(df_list, axis=0).reset_index(drop=True)
print(f'전체 데이터 shape: {df.shape}')
df.head()

### 1. 만족도 (Satisfaction) - KNU 감성사전

In [ ]:
import json

# 감성사전 로딩
with open('./SentiWord_info.json', 'rb') as f:
    sent_dict = json.load(f)

print(f'감성사전 단어 수: {len(sent_dict)}')

In [ ]:
# 감성점수 계산 함수
def sentiment_score(sent_dict, review_token_list):
    result_list = []
    for token in review_token_list:
        for senti_info in sent_dict:
            if token == senti_info['word']:
                result_list.append((senti_info['polarity'], senti_info['word']))
    return result_list

# 테스트
print(sentiment_score(sent_dict, df.loc[0, 'tagged_review']))

In [ ]:
import numpy as np
from tqdm import tqdm

# 전체 감성점수 계산
sentiment = []
for review in tqdm(df['tagged_review']):
    sentiment.append(sentiment_score(sent_dict, review))

# 문서별 평균 감성점수
avg_sentiment_score = []
for rs in sentiment:
    scores = [int(s[0]) for s in rs]
    avg_score = np.mean(scores) if len(scores) > 0 else 0
    avg_sentiment_score.append(avg_score)

df['sentiment_score'] = avg_sentiment_score
df.head()

In [ ]:
from new_value_analysis.opportunity_area_analysis import minmax_scale_scores

# Actor별 Action 감성점수 평균
action_sentiments = {}
for actor in sorted(df['cluster'].unique()):
    actor_df = df[df['cluster'] == actor]
    for action in sorted(actor_df['action_cluster'].unique()):
        action_df = actor_df[actor_df['action_cluster'] == action]
        action_sentiment_avg = np.mean(action_df['sentiment_score'])
        action_sentiments[f'Actor{actor}_Action{action}'] = action_sentiment_avg

# 0~10 정규화
scaled_score = minmax_scale_scores(action_sentiments, feature_range=(0, 10))
for key, score in zip(action_sentiments.keys(), scaled_score):
    action_sentiments[key] = score

satisfaction_df = pd.DataFrame(action_sentiments.items(), columns=['Action', 'satisfaction'])
satisfaction_df.to_pickle('./satisfaction_df.pkl')
print('✅ satisfaction_df.pkl 저장 완료')
satisfaction_df

### 2. 중요도 (Importance) - 언급량 기반

In [ ]:
# Actor_Action 라벨 생성
actor_action_labels = [
    f'Actor{actor}_Action{action}'
    for actor, action in zip(df['cluster'], df['action_cluster'])
]

s = pd.Series(actor_action_labels)
importances = s.value_counts(normalize=True)  # 비율로 계산

# 1~10 정규화
scaled_importances = minmax_scale_scores(importances, feature_range=(1, 10))

importance_df = importances.reset_index()
importance_df.columns = ['Action', 'count']
importance_df['importance'] = scaled_importances
importance_df

### 3. 만족도 + 중요도 병합

In [ ]:
satisfaction_df = pd.read_pickle('./satisfaction_df.pkl')
satisfaction_df = satisfaction_df.set_index('Action')
importance_df = importance_df.set_index('Action')

satisfaction_df['importance'] = importance_df['importance']
DCX_summary = satisfaction_df.copy()
DCX_summary

### 4. Opportunity Score 계산
- 공식: 중요도 + max(중요도 - 만족도, 0)
- 중요도 높고 만족도 낮을수록 기회점수 높음

In [ ]:
def opportunity_score(sat, imp):
    return imp + max(imp - sat, 0)

opp_score_list = [
    opportunity_score(s, i)
    for s, i in zip(DCX_summary['satisfaction'], DCX_summary['importance'])
]

DCX_summary['opportunity'] = opp_score_list
DCX_summary = DCX_summary.reset_index()
DCX_summary.to_csv('./dcx_summary.csv', index=False, encoding='utf-8-sig')

print('기회점수 순위:')
DCX_summary.sort_values('opportunity', ascending=False)

### 5. 기회영역 시각화

In [ ]:
from new_value_analysis.opportunity_area_analysis import plot_opportunity_area, OpportunityPlotConfig

cfg = OpportunityPlotConfig(save_path='./Opportunity_area.png')
plot_opportunity_area(DCX_summary.set_index('Action'), cfg)

In [ ]:
# Underserved Area 출력 (핵심 기회영역)
print('📌 Underserved Area (중요도 높고 만족도 낮은 = 핵심 기회영역):')
underserved = DCX_summary[
    (DCX_summary['importance'] >= DCX_summary['importance'].median()) &
    (DCX_summary['satisfaction'] <= DCX_summary['satisfaction'].median())
].sort_values('opportunity', ascending=False)
print(underserved[['Action', 'importance', 'satisfaction', 'opportunity']])